In [2]:
import pennylane as qml

n_electrons = 4
n_spin_orbitals = 12  # LiH/STO-3G

# 单激发、双激发 轨道索引对/四元组
singles, doubles = qml.qchem.excitations(n_electrons, n_spin_orbitals)

print("=== 单激发项 (i,a) ===")
for s in singles:
    print(s)

print("\n=== 双激发项 (i,j,a,b) ===")
for d in doubles:
    print(d)

=== 单激发项 (i,a) ===
[0, 4]
[0, 6]
[0, 8]
[0, 10]
[1, 5]
[1, 7]
[1, 9]
[1, 11]
[2, 4]
[2, 6]
[2, 8]
[2, 10]
[3, 5]
[3, 7]
[3, 9]
[3, 11]

=== 双激发项 (i,j,a,b) ===
[0, 1, 4, 5]
[0, 1, 4, 7]
[0, 1, 4, 9]
[0, 1, 4, 11]
[0, 1, 5, 6]
[0, 1, 5, 8]
[0, 1, 5, 10]
[0, 1, 6, 7]
[0, 1, 6, 9]
[0, 1, 6, 11]
[0, 1, 7, 8]
[0, 1, 7, 10]
[0, 1, 8, 9]
[0, 1, 8, 11]
[0, 1, 9, 10]
[0, 1, 10, 11]
[0, 2, 4, 6]
[0, 2, 4, 8]
[0, 2, 4, 10]
[0, 2, 6, 8]
[0, 2, 6, 10]
[0, 2, 8, 10]
[0, 3, 4, 5]
[0, 3, 4, 7]
[0, 3, 4, 9]
[0, 3, 4, 11]
[0, 3, 5, 6]
[0, 3, 5, 8]
[0, 3, 5, 10]
[0, 3, 6, 7]
[0, 3, 6, 9]
[0, 3, 6, 11]
[0, 3, 7, 8]
[0, 3, 7, 10]
[0, 3, 8, 9]
[0, 3, 8, 11]
[0, 3, 9, 10]
[0, 3, 10, 11]
[1, 2, 4, 5]
[1, 2, 4, 7]
[1, 2, 4, 9]
[1, 2, 4, 11]
[1, 2, 5, 6]
[1, 2, 5, 8]
[1, 2, 5, 10]
[1, 2, 6, 7]
[1, 2, 6, 9]
[1, 2, 6, 11]
[1, 2, 7, 8]
[1, 2, 7, 10]
[1, 2, 8, 9]
[1, 2, 8, 11]
[1, 2, 9, 10]
[1, 2, 10, 11]
[1, 3, 5, 7]
[1, 3, 5, 9]
[1, 3, 5, 11]
[1, 3, 7, 9]
[1, 3, 7, 11]
[1, 3, 9, 11]
[2, 3, 4, 5]
[2, 3, 4, 7]
[2, 3

In [3]:
import pennylane as qml
import netket as nk
import numpy as np

# ===================== 1. LiH 分子轨道参数 (STO-3G) =====================
n_electrons = 4
n_spin_orbitals = 12    # 总自旋轨道数
n_space_orbitals = n_spin_orbitals // 2  # 空间轨道数 = 6
# 自旋拆分 (aabb 排序): 前6=α, 后6=β
occ_alpha = [0, 1, 2, 3]   # α 占据轨道（基态电子）
occ_beta = []              # β 无占据轨道
unocc = [4,5,6,7,8,9,10,11]# 所有空轨道

# ===================== 2. 提取 CCSD 单/双激发 =====================
singles, doubles = qml.qchem.excitations(n_electrons, n_spin_orbitals)

# ===================== 3. 转换为 NetKet edges (跃迁轨道对) =====================
edges = set()  # 用集合自动去重

# 1) 单激发 → 直接添加边
for i, a in singles:
    edges.add((i, a))

# 2) 双激发 → 拆分为两组单激发边
for i, j, a, b in doubles:
    edges.add((i, a))
    edges.add((j, b))

# 转为列表并排序
edges = sorted(list(edges))
print("=== LiH CCSD 激发对应的 NetKet edges ===")
print(f"总跃迁边数: {len(edges)}")
print("edges =", edges)

# ===================== 4. 初始化 NetKet 希尔伯特空间 (严格 aabb 排序) =====================
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=n_space_orbitals,   # 空间轨道数 6
    s=1/2,
    n_fermions_per_spin=(2, 2),    # α:2电子, β:2电子 (总计4电子，LiH 基态)
)

# ===================== 5. 采样器：使用 CCSD 激发路径作为跳跃规则 =====================
graph = nk.graph.Graph(edges=edges)
hop_rule = nk.sampler.rules.FermionHopRule(hilbert=hi, graph=graph)

# 蒙特卡洛采样器
sampler = nk.sampler.MetropolisSampler(
    hilbert=hi,
    rule=hop_rule,
    n_chains=100,
    sweep_size=32,
)

print("\n=== 希尔伯特空间与采样器初始化完成 ===")
print(f"希尔伯特空间大小: {hi.size}")
print(f"自旋轨道排序: αααααα ββββββ (aabb 标准排序)")

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: With many Markov Chains (e.g GPUs), n_discard_per_chain>5 is often inefficient.

=== LiH CCSD 激发对应的 NetKet edges ===
总跃迁边数: 30
edges = [(0, 4), (0, 5), (0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (1, 4), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (2, 4), (2, 5), (2, 6), (2, 7), (2, 8), (2, 9), (2, 10), (2, 11), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (3, 10), (3, 11)]

=== 希尔伯特空间与采样器初始化完成 ===
希尔伯特空间大小: 12
自旋轨道排序: αααααα ββββββ (aabb 标准排序)


In [6]:
hi.all_states()[0:5]

Array([[0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1],
       [0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1],
       [0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0]], dtype=int8)

In [8]:
import itertools
import netket as nk

# 1. 给定 HF 参考态
hf_state = [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1]

# 2. 提取占据轨道、空轨道
occ = [idx for idx, val in enumerate(hf_state) if val == 1]
virt = [idx for idx, val in enumerate(hf_state) if val == 0]

print("HF 占据轨道 occ =", occ)
print("HF 空轨道 virt =", virt)

# 3. 生成 CCSD 对应的跃迁边（集合自动去重）
edges_set = set()

# -------- 单激发 (i -> a) --------
for i in occ:
    for a in virt:
        edges_set.add((i, a))

# -------- 双激发 (i,j -> a,b) 拆为两条单跃迁 --------
for i, j in itertools.combinations(occ, 2):
    for a, b in itertools.combinations(virt, 2):
        edges_set.add((i, a))
        edges_set.add((j, b))

# 转为有序列表
edges = sorted(list(edges_set))
print(f"总边数: {len(edges)}")
print("edges =", edges)

HF 占据轨道 occ = [4, 5, 10, 11]
HF 空轨道 virt = [0, 1, 2, 3, 6, 7, 8, 9]
总边数: 32
edges = [(4, 0), (4, 1), (4, 2), (4, 3), (4, 6), (4, 7), (4, 8), (4, 9), (5, 0), (5, 1), (5, 2), (5, 3), (5, 6), (5, 7), (5, 8), (5, 9), (10, 0), (10, 1), (10, 2), (10, 3), (10, 6), (10, 7), (10, 8), (10, 9), (11, 0), (11, 1), (11, 2), (11, 3), (11, 6), (11, 7), (11, 8), (11, 9)]
